# A CNN on CIFAR-10: From Digits to Photographs

Every CNN so far has classified handwritten digits — a centered white stroke on a black background,
about the easiest image a network will ever be handed. CIFAR-10 is the step to real photographs:
32×32 color images of ten object classes, where the subject can sit anywhere in the frame, at any
scale, against a cluttered natural background.

The architecture barely changes. What changes is how hard the problem is, and this notebook is
mostly about reading that difficulty honestly — in the loss curves, in which classes the confusion
matrix mixes up, and in what the first convolutional block learns to look for when the input is a
photograph rather than a pen stroke.

## Learning objectives

- Load CIFAR-10 and describe how its shape differs from MNIST's.
- Build a deeper CNN with four convolutional blocks for color input.
- Explain the roles of batch normalization, dropout, and global average pooling in that stack.
- Read a ten-class confusion matrix to identify which categories a model conflates, and why.
- Inspect the first convolutional block's feature maps on a color photograph.

## Background

You should already be able to build, train, and evaluate a small CNN in Keras, and to read a
confusion matrix and classification report.

The one structural difference from MNIST is the channel axis. MNIST images arrive as `(N, 28, 28)`
and need reshaping to `(N, 28, 28, 1)` for a single gray channel. CIFAR-10 arrives as
`(N, 32, 32, 3)` already — the trailing 3 is red, green, and blue — so no reshape is needed, and
the first convolutional layer's kernels now span all three color channels at once.

The other difference is not structural but it matters more: the ten CIFAR classes are semantic
categories rather than glyphs. A "dog" has no canonical shape, so the network cannot succeed by
memorizing a template. It has to build up from local textures and parts, which is what makes this
dataset the right place to look at feature maps.

## This notebook covers

1. Loading CIFAR-10 and viewing sample images
2. Building, training, and evaluating a four-block CNN
3. Visualizing what the first convolutional block responds to
4. Review

**Prerequisites:** `U2-2_CNN-2_MNIST.ipynb` for the CNN architecture and the training helper;
`U2-2_CNN-1_DenseFails.ipynb` for why a dense network stalls on this dataset.

**Dataset:** CIFAR-10, loaded via `tensorflow.keras.datasets.cifar10` (downloads on first use).

**References:** https://www.cs.toronto.edu/~kriz/cifar.html

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math

pd.set_option('display.max_columns',100)
pd.set_option('display.max_rows',100)

plt.style.use('dark_background')

import warnings
warnings.filterwarnings('ignore')

# Shared course helpers (msds565_helpers.py lives in the repo root).
# Notebooks sit two folders below the root, so '../..' points back to it.
import sys
sys.path.append('../..')
import msds565_helpers as helpers

## 1. The CIFAR-10 dataset

### 1.1 Load the data

CIFAR-10 arrives as `(50000, 32, 32, 3)` — fifty thousand 32×32 images with three color channels,
already in the shape a Keras convolution expects.

The labels come back with shape `(N, 1)` rather than `(N,)`, so we flatten them; several
scikit-learn metrics expect a flat array of integers.

In [ ]:
from tensorflow.keras.datasets import cifar10

# Load data
(X_train, y_train), (X_test, y_test) = cifar10.load_data()

# Labels arrive as (N, 1); flatten to (N,) so sklearn's metrics accept them directly.
y_train = y_train.flatten()
y_test  = y_test.flatten()

# CIFAR-10's classes, in label order (0 = airplane, 1 = automobile, ...).
cifar_classes = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                 'dog', 'frog', 'horse', 'ship', 'truck']

# Print shapes
print("X_train.shape:", X_train.shape)
print("X_test.shape: ", X_test.shape)

### 1.2 Look at some images

Worth doing before any modeling. At 32×32 these images are small enough that several classes are
genuinely hard for a person to call — which sets a realistic expectation for what the model can
achieve, and predicts in advance which pairs the confusion matrix will mix up.

In [ ]:
# One example of each of the ten classes.
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for cls, ax in enumerate(axes.ravel()):
    ax.imshow(X_train[np.where(y_train == cls)[0][0]])
    ax.set_title(cifar_classes[cls], fontsize=10)
    ax.axis('off')
plt.suptitle("CIFAR-10 — one example per class")
plt.tight_layout()
plt.show()

# A single random image, at the size the network actually sees it.
idx = np.random.randint(0, len(X_train))
plt.figure(figsize=(4, 4))
plt.imshow(X_train[idx])
plt.title(f"Label: {cifar_classes[y_train[idx]]}")
plt.show()

## 2. Build and train the CNN

### 2.1 Build the model

Four convolutional blocks, each following the same pattern:

`Conv2D` → `BatchNormalization` → `Activation('relu')` → `MaxPooling2D` → `Dropout`

Each piece earns its place:

- **`Conv2D` with a growing filter count** (8 → 16 → 32 → 64). Early layers need few filters
  because there are only so many kinds of edge; deeper layers combine those into many more possible
  textures and parts, so they need more.
- **`BatchNormalization`** standardizes each layer's activations across the batch, which keeps
  gradients well-scaled and lets us train at a relatively aggressive learning rate.
- **`MaxPooling2D`** halves both spatial dimensions, so 32×32 → 16 → 8 → 4 → 2. Two things happen
  at once: the computation shrinks, and each later filter's **receptive field** — the region of the
  original image influencing it — grows. A 3×3 kernel four blocks deep is looking at a large
  fraction of the picture.
- **`Dropout`** randomly zeroes a quarter of the activations during training, forcing the network to
  spread its evidence across features rather than depending on any single one.

The head is `GlobalAveragePooling2D` rather than `Flatten`, collapsing each of the 64 final feature
maps to a single average. That keeps only *whether* a feature was present, not *where* — the right
trade when the object may appear anywhere in the frame — and keeps the classifier tiny at
64 × 10 weights.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import *

dropout_rate = 0.25

n_classes  = np.unique(y_train).shape[0]

# Create model
model = Sequential([
    Input(shape=X_train.shape[1:]),

    Conv2D(8, (3, 3), padding='same'),
    BatchNormalization(),
    Activation('relu'),
    MaxPooling2D(pool_size=(2, 2)),
    Dropout(dropout_rate),
    
    Conv2D(16, (3, 3), padding='same'),
    BatchNormalization(),
    Activation('relu'),
    MaxPooling2D(pool_size=(2, 2)),
    Dropout(dropout_rate),
    
    Conv2D(32, (3, 3), padding='same'),
    BatchNormalization(),
    Activation('relu'),
    MaxPooling2D(pool_size=(2, 2)),
    Dropout(dropout_rate),
    
    Conv2D(64, (3, 3), activation='relu', padding='same'),
    MaxPooling2D(pool_size=(2, 2)),
    
    GlobalAveragePooling2D(),
    #Flatten(),
    
    Dense(n_classes, activation='softmax'),
])

# Display model summary
model.summary()

### 2.2 Compile and configure training

Batch normalization makes a learning rate of 0.01 workable — without it, Adam at this rate would
likely diverge on a stack this deep.

The large batch size (1,000) keeps each epoch fast on 50,000 images, at the cost of fewer gradient
updates per epoch. Early stopping with `restore_best_weights=True` then guarantees we keep the
weights from the best validation epoch rather than whatever the last epoch happened to produce.

In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers.schedules import ExponentialDecay, PolynomialDecay

# define training parameters
epochs          = 15
batch_size      = 1000

# Define the learning rate schedule
initial_learning_rate = 0.01

optimizer = Adam(
    learning_rate=initial_learning_rate,
)

# Compile model
model.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Early stopping callback
early_stopping = EarlyStopping(
    monitor='val_loss',  # Monitor validation loss
    patience=10,          # Stop after 5 epochs without improvement
    restore_best_weights=True  # Restore the best weights after stopping
)

### 2.3 Train and evaluate

Passing `class_names=cifar_classes` labels the confusion matrix axes and the classification report
with real category names instead of the integers 0–9, which makes the interesting pattern much
easier to spot.

Two things to look for in the output:

- **The loss curves.** If validation loss flattens or turns upward while training loss keeps
  falling, the model has started memorizing rather than generalizing — far more likely here than on
  MNIST.
- **Which classes get confused.** The errors will not be spread evenly. Expect the animal classes,
  especially cat and dog, to trade predictions much more than, say, ship and frog. Similar shapes
  and textures at 32×32 leave genuinely little to distinguish them.

In [ ]:
model, history = helpers.train_and_evaluate(
    model,
    X_train, y_train,
    X_test, y_test,
    epochs=epochs,
    batch_size=batch_size,
    callbacks=[early_stopping],
    class_names=cifar_classes
)

## 3. Visualize channels after processing

The same feature-map view used on MNIST, now on a color photograph — and the difference is
instructive.

We read activations at layer index 2, which is the `Activation('relu')` of the first block
(`Conv2D` = 0, `BatchNormalization` = 1, `Activation` = 2). Because ReLU has already zeroed the
negative responses, each map shows only where that filter positively fired.

What to look for: on MNIST the first block learned stroke and edge detectors. Here, with three
input channels to work with, expect some filters to key on **color** contrast rather than
brightness edges — one map lighting up on the sky, another on the object, a third on a particular
texture. The kernels now span all three color channels, so color opponency is available to them in
a way it simply was not on gray-scale digits.

Re-run this cell a few times to see different images, and try a deeper `n` (say 6 or 10) to watch
the maps become smaller, more abstract, and much harder to interpret — which is exactly why
`U2-2_CNN-8_Explainability.ipynb` exists.

In [ ]:
idx = np.random.choice( range(X_train.shape[0]), 1 )[0]

# Choose an image from your dataset
sample_img = X_train[idx]  # or any image shaped like your input

plt.figure(figsize=(3, 3))
plt.imshow(sample_img)
plt.title(f"Feeding in: {cifar_classes[y_train[idx]]}")
plt.axis('off')
plt.show()

In [ ]:
# Visualize the first conv block's activation output (Conv2D=0, BatchNormalization=1, Activation=2)
helpers.visualize_layer_outputs(model, sample_img, n=2)

## 4. Review

| | MNIST (`U2-2_CNN-2`) | CIFAR-10 (this notebook) |
|---|---|---|
| Input shape | `(N, 28, 28, 1)` after reshape | `(N, 32, 32, 3)` as loaded |
| Content | Centered white glyph, black background | Object anywhere in a natural scene |
| Conv blocks | 2 | 4, with filters growing 8 → 16 → 32 → 64 |
| Head | `Flatten` or `GlobalAveragePooling2D` | `GlobalAveragePooling2D` |
| Typical accuracy | ~0.98 | Far lower — and that is the honest result |

**Takeaways**

- **The architecture barely changed; the difficulty did.** Nearly the same layer stack that nearly
  saturates MNIST lands much lower here. The gap is not a bug in the model — it measures how much
  harder semantic categories in cluttered scenes are than centered glyphs.
- **Depth buys receptive field.** Each pooling step halves the spatial dimensions, so a fixed 3×3
  kernel sees an ever-larger share of the original image as you go deeper. That is how a network
  built entirely from *local* operations ends up making a *global* decision.
- **Read the confusion matrix structurally, not just diagonally.** The off-diagonal mass clusters —
  animals with animals, vehicles with vehicles. That pattern says the model has learned something
  real about visual similarity, and it separates "the model is broken" from "these classes look
  alike at 32×32."
- **Color changes what the first layer learns.** With three input channels, first-block filters can
  respond to color contrast, not only to brightness edges — an option that did not exist on
  gray-scale digits.
- **This is where accuracy stops being enough.** Once a model is wrong a meaningful fraction of the
  time, "which classes, and on what evidence?" becomes the more useful question — the thread picked
  up in `U2-2_CNN-8_Explainability.ipynb`.

**Where to take it further:** the obvious next levers are data augmentation (as in
`U2-2_CNN-2_MNIST.ipynb`), normalizing pixels to $[0,1]$, and a longer training schedule. The much
larger lever is not to train from scratch at all — see `U2-2_CNN-6_TransferLearning.ipynb`.

**Next:** `U2-2_CNN-5_Multimodal.ipynb` combines image input with tabular features in a single
model.